# form-correction: cloud runner

Cloud entry point. Cells follow ARCHITECTURE.md section 6.

---

## Expected result (read before you panic)

On the clean 2026-11-25 run (KIMORE Ex5, 75 rows, Gemini backend live), the primary ridge-on-flags detector returned **Spearman rho = -0.52 on held-out subjects (95% CI -0.77 to -0.16, n_test = 23)**, with MAE 0.075 vs mean-baseline MAE 0.069. **This negative result is the published finding of the creative component, not a bug.** The flags as currently combined predict the clinician score *backwards*, consistently (CI never crosses zero); see `DECISIONS.md` section 5 (2026-11-25 second entry) and `PAPER_DRAFT.md` section 4.2. Re-running this notebook on the same KIMORE pickle with `RANDOM_SEED` will reproduce rho in roughly that interval.

Multi-view angle stability is fine (rho between -0.54 and -0.47 across 0 to 60 degrees; mean flag_error 1.8 to 2.6 degrees at 15 to 60 degrees), so the inversion is a feature problem, not a rotation problem. The grounded verifier and refusal path still work as designed.


## 1. Setup
Install dependencies and set deterministic seeds.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/wizwall-anwar/form-correction.git"
if os.path.isdir("/kaggle"):
    REPO_DIR = "/kaggle/working/form-correction"
elif os.path.isdir("/content"):
    REPO_DIR = "/content/form-correction"
else:
    REPO_DIR = os.getcwd()

try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        get_ipython().system(f"git -C {REPO_DIR} pull --ff-only")
    elif not os.path.isdir(REPO_DIR):
        get_ipython().system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

try:
    get_ipython().system("pip install -q -r requirements.txt")
except Exception:
    pass

import random
import numpy as np
from src.config import RANDOM_SEED
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"[Setup] done (REPO_DIR={REPO_DIR!r})")

## 2. Config
Set `DATA_DIR` to a manual dataset path to skip fetching.

In [ ]:
import importlib
import os
from src.config import resolve_data_dir

# --- Gemini API key for VLM agreement (cell 6) ---
# Option 1: Colab Secrets sidebar > key icon > add GEMINI_API_KEY
# Option 2: Kaggle > Add-ons > Secrets > add GEMINI_API_KEY
_providers = (
    ('Colab', 'google.colab.userdata', lambda module: module.get('GEMINI_API_KEY')),
    ('Kaggle', 'kaggle_secrets', lambda module: module.UserSecretsClient().get_secret('GEMINI_API_KEY')),
)
_key = None
for _provider, _module_name, _read_secret in _providers:
    try:
        _key = _read_secret(importlib.import_module(_module_name))
    except ImportError:
        continue
    except Exception as exc:
        print(f'[Config] {_provider} secret read failed: {type(exc).__name__}: {exc}')
        print("[Config] Hint: secret must be named exactly GEMINI_API_KEY; on Colab turn on 'Notebook access' for it; on Kaggle attach it under Add-ons > Secrets")
        continue
    if _key:
        os.environ['GEMINI_API_KEY'] = _key
        os.environ['VLM_BACKEND'] = 'gemini'
        print(f'[Config] Gemini key loaded from {_provider} ({len(_key)} chars)')
        break
if not _key:
    os.environ['VLM_BACKEND'] = 'placeholder'
    print('[Config] VLM_BACKEND=placeholder (offline stub). Agreement kappas will NOT be real VLM results.')

DATA_DIR = resolve_data_dir(None)
print(f'[Config] DATA_DIR={DATA_DIR!r}' if DATA_DIR else '[Config] no local dataset found; synthetic fallback will run')


## 3. Data
Load through the bounded fallback chain, using synthetic squats if loading fails.

In [ ]:
from src import m00_data
data_is_synthetic = False
try:
    samples = m00_data.load_data(DATA_DIR)
    data_source = samples[0].data_source if samples else "synthetic"
    data_is_synthetic = data_source == "synthetic"
except Exception:
    samples = m00_data.make_synthetic_squats(20, RANDOM_SEED)
    data_source = "synthetic"
    data_is_synthetic = True
print(f"[Data] {data_source} ({len(samples)} samples)")

## 4. Features
Create rotated, jittered synthetic views and compute kinematic flags.

In [ ]:
from src.config import JITTER_SIGMA, ROTATION_ANGLES_DEG
from src.m01_features import add_jitter, kinematic_flags, rotate_view
views_by_angle = {}
flags_by_angle = {}
try:
    for angle in ROTATION_ANGLES_DEG:
        views_by_angle[angle] = []
        flags_by_angle[angle] = []
        for index, sample in enumerate(samples):
            joints = add_jitter(rotate_view(sample.joints, angle), JITTER_SIGMA, RANDOM_SEED + index + int(angle))
            views_by_angle[angle].append(joints)
            flags_by_angle[angle].append(kinematic_flags(joints))
except Exception:
    views_by_angle = {0: [sample.joints for sample in samples]}
    flags_by_angle = {0: [kinematic_flags(sample.joints) for sample in samples]}
print(f"[Features] {len(flags_by_angle)} view angles")

## 5. Detector
Run splits, baseline training, calibration, abstention, and metrics.

In [ ]:
from src import m02_detector
from src.m03_agreement import detector_vs_clinician_kappa

detector_result = {}
predictions = []
risk = {'coverage': [], 'risk': []}
detector_metrics = {'spearman_rho': 0.0, 'mae': 0.0, 'ece': 0.0, 'n_test': 0}
clip_metrics = {'spearman_rho': 0.0, 'mae': 0.0, 'n_test': 0}
table2_rows = []
per_flag_subject_split = {}
dvc_kappa = {'n_scored': 0, 'cohen_kappa': None, 'quadratic_weighted_cohen_kappa': None}
try:
    detector_result = m02_detector.run_evaluation(samples, flags_by_angle, RANDOM_SEED)
    predictions = detector_result['predictions']
    risk = detector_result['risk_coverage']
    detector_metrics = {**detector_result['subject_split'], 'ece': detector_result['ece']}
    clip_metrics = detector_result['clip_split']
    table2_rows = detector_result['table2_rows']
    per_flag_subject_split = detector_result.get('per_flag_subject_split', {})
    # Detector-vs-clinician kappa on all 75 samples using dataset-level tertile bins.
    # The detector trained with train-only tertile edges, so this is a whole-dataset
    # reliability number, not a held-out comparison; the paper notes the mismatch.
    clinician_bins_dataset = [int(s.clinician_bin) for s in samples]
    dvc_kappa = detector_vs_clinician_kappa(
        [p['pred_bin'] for p in predictions],
        clinician_bins_dataset,
    )
    print(f'[Detector] rho={detector_metrics["spearman_rho"]:.3f}, MAE={detector_metrics["mae"]:.3f}, n_test={detector_metrics["n_test"]}')
    print(f'[Detector] per-flag rho vs cTS (held-out subjects): {per_flag_subject_split}')
    print(f'[Detector] detector-vs-clinician kappa (all samples): {dvc_kappa}')
except Exception as exc:
    predictions = [{'pred_bin': int(sample.clinician_bin), 'confidence': 1.0, 'abstain_flag': False} for sample in samples]
    print(f'[Detector] error: {exc}')

# Build detector records for the demo
detector_records = []
base_flags = flags_by_angle.get(0, [])
for index, sample in enumerate(samples):
    flags = base_flags[index] if index < len(base_flags) else m02_detector.kinematic_flags(sample.joints)
    pred = predictions[index] if index < len(predictions) else {'pred_bin': int(sample.clinician_bin), 'confidence': 1.0, 'abstain_flag': False}
    detector_records.append({'flags': flags, 'view_deg': 0.0, **pred})


## 6. Agreement
Compare the offline VLM stub with clinician bins at 0 and 45 degrees.

In [ ]:
from src.m03_agreement import agreement_report
try:
    backend = os.environ.get('VLM_BACKEND', 'placeholder')
    print(f'[Agreement] backend={backend}')
    agreement = agreement_report(samples, view_degs=(0, 45), seed=RANDOM_SEED, backend=backend)
    for deg, metrics in agreement.get('model_vs_clinician_reliability', {}).items():
        n = metrics.get('sample_count', 0)
        n_scored = metrics.get('n_scored', 0)
        n_stub = metrics.get('n_stub', 0)
        kappa = metrics.get('cohen_kappa')
        qw = metrics.get('quadratic_weighted_cohen_kappa')
        kappa_text = 'unavailable' if kappa is None else f'{kappa:.3f}'
        qw_text = 'unavailable' if qw is None else f'{qw:.3f}'
        print(f'  {deg} deg: n={n}, n_scored={n_scored}, n_stub={n_stub}, kappa={kappa_text}, qw_kappa={qw_text}')
        if backend == 'gemini' and n_scored == 0:
            print(f'  WARNING: GEMINI CALL FAILED FOR {deg} DEG; KAPPA IS UNAVAILABLE.')
except Exception as exc:
    agreement = {'model_vs_clinician_reliability': {}}
    print(f'[Agreement] error: {exc}')
print('[Agreement] done')


## 7. Hallucination comparison
Log grounded demo output against a deliberately free-form comparison string.

In [ ]:
from app.biocoach_demo import log_hallucination_comparison, run_demo
hallucination_logs = []
try:
    free_form_text = "The knee flexion is 999.0 degrees and trunk lean is 888.0 degrees."
    for detector_record in detector_records[:2]:
        grounded = run_demo(detector_record)
        hallucination_logs.append(log_hallucination_comparison(free_form_text, grounded, detector_record))
except Exception:
    hallucination_logs = []
hallucination_rates = {
    "free_form": float(np.mean([item["free_form_unsupported_count"] for item in hallucination_logs])) if hallucination_logs else 0.0,
    "grounded": float(np.mean([item["grounded_unsupported_count"] for item in hallucination_logs])) if hallucination_logs else 0.0,
    "note": "free_form is a PLANTED fabricated control string (999/888), not VLM output; this is a verifier check, not a measured hallucination rate.",
}
print("[Hallucination] done")

## 8. Report
Package tables, the risk-coverage figure, manifest, and results bundle.

In [ ]:
import json as _json
from pathlib import Path as _Path
from src.m04_report import (build_results_bundle, build_table1, build_table2,
                            build_table3, build_table4, make_risk_coverage_figure,
                            save_manifest, stamp_synthetic)

try:
    # Use held-out metrics from the Detector cell
    if not table2_rows:
        table2_rows = [{'angle': a, 'spearman_rho': 0.0, 'mae': 0.0, 'flag_error': 0.0} for a in flags_by_angle]

    table1 = build_table1(data_source, detector_result.get('n_subjects', len({s.subject_id for s in samples})),
                          detector_result.get('n_sequences', len(samples)), detector_metrics, clip_metrics,
                          detector_result.get('max_sequences_per_subject', 0),
                          per_flag_subject_split=per_flag_subject_split)
    table2 = build_table2(table2_rows)
    table3 = build_table3(detector_metrics.get('ece', 0.0),
                          detector_result.get('selective_accuracy_at_coverage', {}), risk,
                          detector_metrics.get('n_test'), detector_result.get('test_class_counts'))
    table4 = build_table4(agreement, hallucination_rates, detector_vs_clinician=dvc_kappa)
    table1, table2, table3, table4 = [stamp_synthetic(t, data_is_synthetic) for t in (table1, table2, table3, table4)]

    # Save tables as JSON so they end up in the zip
    _Path('results').mkdir(parents=True, exist_ok=True)
    for name, tbl in [('table1', table1), ('table2', table2), ('table3', table3), ('table4', table4)]:
        _Path(f'results/{name}.json').write_text(_json.dumps(tbl, indent=2, default=str))

    make_risk_coverage_figure(risk.get('coverage', []), risk.get('risk', []))
    save_manifest(data_source, RANDOM_SEED)
    bundle_path = build_results_bundle()

    # Print all tables
    for name in ('table1', 'table2', 'table3', 'table4'):
        content = _json.loads(_Path(f'results/{name}.json').read_text())
        print(f'\n=== {name} ===')
        print(_json.dumps(content, indent=2, default=str))

    # Download
    if os.path.isdir('/content') and not os.path.isdir('/kaggle'):
        try:
            from google.colab import files
            files.download(bundle_path)
        except Exception:
            pass
    elif os.path.isdir('/kaggle'):
        import shutil
        shutil.copy2(bundle_path, '/kaggle/working/results_bundle.zip')
        print(f'[Report] copied to /kaggle/working/results_bundle.zip')
    print(f'[Report] complete: {bundle_path}')
except Exception as exc:
    bundle_path = None
    print(f'[Report] error: {exc}')
    import traceback; traceback.print_exc()
